<a href="https://colab.research.google.com/github/mohammedAlkhuzaie/Suha-Ali-Salman/blob/main/Test_document_editor_Suha.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
Test Document Editor — Self-Contained Version
===============================================

"""

from __future__ import annotations

from abc import ABC, abstractmethod
from typing import Callable, Dict, List, Optional

import pytest


# ---------------------------------------------------------------------------
# Document Editor implementation (Factory Method + registry pattern)
# ---------------------------------------------------------------------------

class Document(ABC):
    """Abstract contract that every document format must implement."""

    def __init__(self, content: str = "") -> None:
        self.content = content

    @abstractmethod
    def save(self, path: str) -> str:
        raise NotImplementedError

    @abstractmethod
    def render(self) -> str:
        raise NotImplementedError

    @property
    @abstractmethod
    def extension(self) -> str:
        raise NotImplementedError


class PDFDocument(Document):
    def save(self, path: str) -> str:
        return f"{path}{self.extension}"

    def render(self) -> str:
        return f"[PDF] {self.content}"

    @property
    def extension(self) -> str:
        return ".pdf"


class WordDocument(Document):
    def save(self, path: str) -> str:
        return f"{path}{self.extension}"

    def render(self) -> str:
        return f"[WORD] {self.content}"

    @property
    def extension(self) -> str:
        return ".docx"


class HTMLDocument(Document):
    def save(self, path: str) -> str:
        return f"{path}{self.extension}"

    def render(self) -> str:
        return f"<html><body>{self.content}</body></html>"

    @property
    def extension(self) -> str:
        return ".html"


class UnsupportedFormatError(ValueError):
    """Raised when the editor is asked for a format that has not been registered."""


class DocumentFactory:
    """Registry-based factory. New formats register themselves; the factory
    and the editor never need an if/elif chain or any modification."""

    _registry: Dict[str, Callable[[str], Document]] = {}

    @classmethod
    def register(cls, format_name: str, creator: Callable[[str], Document]) -> None:
        cls._registry[format_name.lower()] = creator

    @classmethod
    def create(cls, format_name: str, content: str = "") -> Document:
        key = format_name.lower()
        if key not in cls._registry:
            raise UnsupportedFormatError(
                f"No document type registered for format '{format_name}'. "
                f"Available: {sorted(cls._registry)}"
            )
        return cls._registry[key](content)

    @classmethod
    def available_formats(cls) -> List[str]:
        return sorted(cls._registry)


DocumentFactory.register("pdf", PDFDocument)
DocumentFactory.register("word", WordDocument)
DocumentFactory.register("html", HTMLDocument)


class DocumentEditor:
    """Core editor logic. Depends only on the Document abstraction and the
    factory -- never on a concrete format class."""

    def __init__(self) -> None:
        self._document: Optional[Document] = None

    def new_document(self, format_name: str, content: str = "") -> Document:
        self._document = DocumentFactory.create(format_name, content)
        return self._document

    def edit(self, content: str) -> None:
        if self._document is None:
            raise RuntimeError("No document open. Call new_document() first.")
        self._document.content = content

    def display(self) -> str:
        if self._document is None:
            raise RuntimeError("No document open. Call new_document() first.")
        return self._document.render()

    def save(self, path: str) -> str:
        if self._document is None:
            raise RuntimeError("No document open. Call new_document() first.")
        return self._document.save(path)

    @property
    def current_document(self) -> Optional[Document]:
        return self._document


# ---------------------------------------------------------------------------
# Tests
# ---------------------------------------------------------------------------

class MarkdownDocument(Document):
    """A brand-new format, defined entirely outside the core implementation,
    to prove new formats can be added without modifying the editor or
    factory."""

    def save(self, path: str) -> str:
        return f"{path}{self.extension}"

    def render(self) -> str:
        return f"# {self.content}"

    @property
    def extension(self) -> str:
        return ".md"


def test_pdf_document_render_and_save():
    doc = PDFDocument("hello")
    assert doc.render() == "[PDF] hello"
    assert doc.save("/tmp/x") == "/tmp/x.pdf"


def test_word_document_render_and_save():
    doc = WordDocument("hello")
    assert doc.render() == "[WORD] hello"
    assert doc.save("/tmp/x") == "/tmp/x.docx"


def test_html_document_render_and_save():
    doc = HTMLDocument("hello")
    assert doc.render() == "<html><body>hello</body></html>"
    assert doc.save("/tmp/x") == "/tmp/x.html"


def test_factory_creates_registered_formats():
    for fmt, cls in (("pdf", PDFDocument), ("word", WordDocument), ("html", HTMLDocument)):
        doc = DocumentFactory.create(fmt, "content")
        assert isinstance(doc, cls)


def test_factory_is_case_insensitive():
    doc = DocumentFactory.create("PDF", "content")
    assert isinstance(doc, PDFDocument)


def test_factory_unsupported_format_raises():
    with pytest.raises(UnsupportedFormatError):
        DocumentFactory.create("fax", "content")


def test_factory_supports_new_format_without_modifying_factory_or_editor():
    DocumentFactory.register("markdown", MarkdownDocument)
    try:
        editor = DocumentEditor()
        editor.new_document("markdown", "Notes")
        assert editor.display() == "# Notes"
        assert "markdown" in DocumentFactory.available_formats()
    finally:
        del DocumentFactory._registry["markdown"]


def test_editor_full_workflow():
    editor = DocumentEditor()
    editor.new_document("html", "draft")
    editor.edit("final content")
    assert editor.display() == "<html><body>final content</body></html>"
    assert editor.save("/tmp/report") == "/tmp/report.html"
    assert isinstance(editor.current_document, HTMLDocument)


def test_editor_raises_when_no_document_open():
    editor = DocumentEditor()
    with pytest.raises(RuntimeError):
        editor.display()
    with pytest.raises(RuntimeError):
        editor.save("/tmp/x")
    with pytest.raises(RuntimeError):
        editor.edit("x")


def test_editor_current_document_none_initially():
    editor = DocumentEditor()
    assert editor.current_document is None


if __name__ == "__main__":
    # Two execution contexts are supported here:
    #
    # 1. Real script (`python test_document_editor_standalone.py`):
    #    `__file__` exists and pytest can discover + run the tests in it
    #    normally via pytest.main([__file__]).
    #
    # 2. Pasted into a Colab/Jupyter cell:
    #    `__file__` does not exist, AND even if we skipped that, pytest
    #    discovers tests from files on disk -- it cannot see functions that
    #    only exist as in-memory objects in the notebook's namespace. So in
    #    this case we just call each `test_*` function directly and report
    #    pass/fail ourselves, with no dependency on pytest's file discovery.
    import sys

    try:
        _this_file = __file__
    except NameError:
        _this_file = None

    if _this_file:
        sys.exit(pytest.main([_this_file, "-v"]))
    else:
        test_functions = {
            name: obj
            for name, obj in list(globals().items())
            if name.startswith("test_") and callable(obj)
        }
        passed, failed = 0, []
        for name, fn in test_functions.items():
            try:
                fn()
                passed += 1
                print(f"PASSED  {name}")
            except Exception as exc:  # noqa: BLE001 - surfacing any failure
                failed.append((name, exc))
                print(f"FAILED  {name}  ->  {exc!r}")
        print(f"\n{passed} passed, {len(failed)} failed out of {len(test_functions)} tests")


PASSED  test_pdf_document_render_and_save
PASSED  test_word_document_render_and_save
PASSED  test_html_document_render_and_save
PASSED  test_factory_creates_registered_formats
PASSED  test_factory_is_case_insensitive
PASSED  test_factory_unsupported_format_raises
PASSED  test_factory_supports_new_format_without_modifying_factory_or_editor
PASSED  test_editor_full_workflow
PASSED  test_editor_raises_when_no_document_open
PASSED  test_editor_current_document_none_initially

10 passed, 0 failed out of 10 tests
